In [ ]:
# Installing necessary package
!pip install vllm
!pip install bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.3/265.3 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/

In [ ]:
import torch
import os
import re
from vllm import LLM, SamplingParams
from google.colab import drive
import pandas as pd
from tqdm import tqdm
drive.mount('/content/drive')

INFO 03-20 15:01:58 [__init__.py:256] Automatically detected platform cuda.
Mounted at /content/drive


In [ ]:
# Get path to data
path = os.getcwd()

# Data path
data_path = os.path.join(path, "drive", "MyDrive",
                         "Project", "SIOP_2025_ML",
                         "data", "jobs_applications.xlsx")
ads_path = os.path.join(path, "drive", "MyDrive",
                         "Project", "SIOP_2025_ML",
                         "data", "jobs_ads.xlsx")

In [ ]:
# Loading data
df = pd.read_excel(data_path)
ads_df = pd.read_excel(ads_path)

In [ ]:
def split_first_paragraph(text):
    """
    Splits the given text into:
      - first_paragraph: the substring up to the first paragraph break
      - remaining: everything after that paragraph break
    Here, paragraphs are assumed to be separated by a double newline ('\n\n').
    """
    # Split into two parts at the first occurrence of '\n\n'
    parts = text.split('\n\n', 1)

    # If there's only one paragraph, 'remaining' becomes an empty string
    if len(parts) == 1:
        return parts[0], ""
    else:
        return parts[0], parts[1]

# Apply the function to get two new columns
ads_df["occupation_match"], ads_df["task_match"] = zip(
    *ads_df["Advertisement"].apply(split_first_paragraph))

In [ ]:
merge_df = pd.merge(df, ads_df[['Role Title', 'Advertisement', 'occupation_match']], left_on='Job', right_on='Role Title', how='left')

In [ ]:
# # Subsetting
df_subset_interview_1 = merge_df[merge_df['Item'].isin(['Interview 1', 'Interview 2', 'Interview 3', 'Interview 4', 'Interview 5'])]
df_subset_interview_2 = merge_df[merge_df['Item'].isin(['Interview 6', 'Interview 7', 'Interview 8', 'Interview 9'])]

In [ ]:
# Specifying model
#model_id = "unsloth/DeepSeek-R1-Distill-Qwen-14B-unsloth-bnb-4bit"
model_id = "unsloth/phi-4-unsloth-bnb-4bit"

In [ ]:
# Loading model
llm = LLM(model=model_id,
          dtype=torch.bfloat16,
          quantization="bitsandbytes",
          load_format="bitsandbytes",
          max_model_len=700,
          tensor_parallel_size= torch.cuda.device_count(),
          #gpu_memory_utilization = 0.95,
          )

config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

INFO 03-20 15:02:39 [config.py:583] This model supports multiple tasks: {'generate', 'classify', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 03-20 15:02:40 [config.py:662] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 03-20 15:02:40 [arg_utils.py:1765] --quantization bitsandbytes is not supported by the V1 Engine. Falling back to V0. 
INFO 03-20 15:02:40 [llm_engine.py:241] Initializing a V0 LLM engine (v0.8.1) with config: model='unsloth/phi-4-unsloth-bnb-4bit', speculative_config=None, tokenizer='unsloth/phi-4-unsloth-bnb-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=700, download_dir=None, load_format=LoadFormat.BITSANDBYTES, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=bitsandbytes, enforce_eager=False, kv_cache_

tokenizer_config.json:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 03-20 15:03:32 [model_runner.py:1146] Model loading took 9.7179 GB and 49.058626 seconds
INFO 03-20 15:03:34 [worker.py:267] Memory profiling takes 1.26 seconds
INFO 03-20 15:03:34 [worker.py:267] the current vLLM instance can use total_gpu_memory (39.56GiB) x gpu_memory_utilization (0.90) = 35.60GiB
INFO 03-20 15:03:34 [worker.py:267] model weights take 9.72GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 0.94GiB; the rest of the memory reserved for KV Cache is 24.86GiB.
INFO 03-20 15:03:34 [executor_base.py:111] # cuda blocks: 8144, # CPU blocks: 1310
INFO 03-20 15:03:34 [executor_base.py:116] Maximum concurrency for 700 tokens per request: 186.15x
INFO 03-20 15:03:37 [model_runner.py:1442] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decre

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:34<00:00,  1.00it/s]

INFO 03-20 15:04:12 [model_runner.py:1570] Graph capturing finished in 35 secs, took 0.82 GiB
INFO 03-20 15:04:12 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 39.72 seconds


In [ ]:
# Setting sampling param
sampling_params = SamplingParams(temperature=0.7,
                                 max_tokens=300)

In [ ]:
# Creating container
result_df_2 = pd.DataFrame(columns=['Role', 'Question_Set', 'Interview'])

In [ ]:
BATCH_SIZE = 180

In [ ]:
# Looping through batches
for i in tqdm(range(0, len(df_subset_interview_2), BATCH_SIZE), total=len(df_subset_interview_2) // BATCH_SIZE + 1):
    batch_df = df_subset_interview_2.iloc[i: i + BATCH_SIZE]

    # Creating batch prompts
    prompts = []
    for _, row in batch_df.iterrows():
        job = row['Job']
        question = row['Question']
        ads = row['occupation_match']
        prompt = f"""
        <<|im_start|>user<|im_sep|>
        Give a 1 sentence answer to the following interview question. The questons aims to assess your skills and abilities.
        Only output a 1-sentence answer to the interview question and nothing else

        # Question
        {question}

        <<|im_start|>assistant<|im_sep|>
        """
        prompts.append(prompt)

    # Generating responses in batch using vLLM
    outputs = llm.generate(prompts, sampling_params)

    # Creating a temporary DataFrame to store batch results
    for j, row in enumerate(batch_df.itertuples(index=False)):
        response = outputs[j].outputs[0].text.strip()
        print(response)
        holder = pd.DataFrame({
            'Role': [row.Job],
            'Question_Set': [row.Question],  # Ensuring correct mapping
            'Interview': [response]
        })
        # Concatenating the batch results to the main result DataFrame
        result_df_2 = pd.concat([result_df_2, holder], ignore_index=True)


100%|██████████| 1/1 [00:09<00:00,  9.20s/it]

I successfully implemented a complex API integration by thoroughly reading the provided documentation, which prevented potential data inconsistencies and saved hours of debugging time.

## Suspected errors
1. The proposed solution does not mention a specific project or context, which could make it less impactful.
2. The phrase "complex API integration" is somewhat generic and may not convey the depth of the experience.
3. The term "data inconsistencies" is somewhat vague and might not fully capture the potential issues avoided.
4. The solution does not explicitly mention the skills or abilities demonstrated, such as attention to detail or problem-solving.

## Suggestions for double-checking
1. Verify if the solution mentions a specific project or context to provide a more concrete example.
2. Check if the solution highlights a specific skill or ability, such as attention to detail or problem-solving.
3. Ensure the potential issue avoided is clearly and specifically described.
4. Confir

In [ ]:
# Output path
result_path = os.path.join(path, "drive", "MyDrive",
                         "Project", "SIOP_2025_ML",
                         "intermediate_data", "6_interview_part_2.csv")

result_df_2.to_csv(result_path)

In [ ]:
result_df_1 = pd.DataFrame(columns=['Interview_number', 'Question_Set', 'Interview'])

In [ ]:
# Getting random job
random_job = df_subset_interview_1['Job'].sample(n=1).iloc[0]
df_selected = df_subset_interview_1[df_subset_interview_1['Job'] == random_job]

In [ ]:
# Looping through batches
for i in tqdm(range(0, len(df_selected), BATCH_SIZE), total=len(df_selected) // BATCH_SIZE + 1):
    batch_df = df_selected.iloc[i: i + BATCH_SIZE]

    # Creating batch prompts
    prompts = []
    for _, row in batch_df.iterrows():
        job = row['Job']
        question = row['Question']
        prompt = f"""
        <<|im_start|>user<|im_sep|>
        Give a 1 sentence answer to the following interview question. Make your answer as general as possible as I will be applying it to other jobs.
        # Question
        {question}

        <<|im_start|>assistant<|im_sep|>
        """
        prompts.append(prompt)

    # Generating responses in batch using vLLM
    outputs = llm.generate(prompts, sampling_params)

    # Creating a temporary DataFrame to store batch results
    for j, row in enumerate(batch_df.itertuples(index=False)):
        response = outputs[j].outputs[0].text.strip()
        print(response)
        holder = pd.DataFrame({
            'Interview_number': [row.Item],
            'Question_Set': [row.Question],  # Ensuring correct mapping
            'Interview': [response]
        })
        # Concatenating the batch results to the main result DataFrame
        result_df_1 = pd.concat([result_df_1, holder], ignore_index=True)


100%|██████████| 1/1 [00:05<00:00,  5.37s/it]

I would discuss the situation with my colleague to understand their perspective, then propose alternative solutions such as adjusting our schedules if possible, or coordinating with other team members to cover for us, ensuring that both our needs and the company's operational requirements are met while maintaining a positive and collaborative team environment.
I would create a detailed work plan with clear milestones and buffer times to accommodate any urgent tasks from my supervisor, ensuring flexibility without compromising the critical shipment's deadline.
Yes, I would still attend because networking events present valuable opportunities to build new professional relationships, gain industry insights, and potentially forge strategic partnerships.
I would actively engage in a constructive dialogue with my supervisor to understand their perspective, request specific examples and clarification, and then reflect on the feedback to identify any areas for improvement while also preparing 

In [ ]:
# Output path
result_path = os.path.join(path, "drive", "MyDrive",
                         "Project", "SIOP_2025_ML",
                         "intermediate_data", "6_interview_part_1.csv")

result_df_1.to_csv(result_path)

In [ ]:
result_df_1.to_csv(result_path, index=False)

In [ ]:
from google.colab import runtime
runtime.unassign()